# RiverSentinel

**How this maps to the team**
- **PAIR 1 — Preprocessing:** turns raw downloads into a clean vector buffer
  + a clipped satellite image, for every region in `REGIONS`
- **PAIR 2 — Modelling:** turns that image into pixel training data, a
  Random Forest baseline, and real per-region structure counts
- **PAIR 3 — UI:** pools every region's real structures into one table and
  puts it on a map (not built in this notebook — this notebook produces
  exactly what Pair 3 needs to start)

**The one rule that keeps three pairs from breaking each other's work:**
every section's OUTPUT file name must exactly match the next section's INPUT
file name. Watch for the `# HANDOFF:` comments.

## Before you write any code: the GitHub file-size problem

Some of the files this pipeline produces do not belong in your Git
repository.

**The actual numbers GitHub enforces:**
- Soft warning at 50MB per file
- Hard block at 100MB per file — the push just fails
- No hard cap on total repo size, but you'll get nudged around 5GB

**What in this project is likely to hit that:**
- The raw Sentinel-2 composite (a multi-band GeoTIFF, even over a small area)
  — often tens to hundreds of MB
- The full folder of image tiles once Pair 1 finishes tiling — hundreds of
  small PNGs adds up fast
- A custom-trained YOLOv8-seg checkpoint — a few MB for the "nano" model,
  but can balloon past 100MB on larger model variants

**Two ways to handle it — this notebook uses the first one:**

1. **Don't commit the data at all.** Put `data/` and `models/` in
   `.gitignore`. Instead, commit the *code that regenerates the data* — the
   Earth Engine export call, the shapefile download, the tiling function.
   Anyone who clones the repo runs the setup cells once and gets the same
   files locally. This is the standard pattern for any project with
   downloadable/reproducible data.
2. **Git LFS** (Large File Storage) — for the rare file you genuinely want
   version-controlled (e.g. "this exact trained model is what shipped"). It
   swaps the real bytes for a small pointer file in Git and stores the
   content separately. Free GitHub accounts get 1GB of LFS storage — plenty
   for one or two checkpoints, not for raw imagery.

We'll use approach 1 as the default below, and call out the one place
(the final YOLO checkpoint) where approach 2 might be worth it.

In [5]:
# This cell writes a .gitignore for the project. Run it once, from the repo root.
# Everything listed here gets regenerated by this notebook -- nothing here is
# hand-crafted, so there's no loss in not tracking it with Git.
import os

gitignore_contents = """
# Raw + processed geospatial data -- regenerate by running this notebook
data/raw/
data/processed/
data/vectors/
data/tiles/

# Trained model weights -- regenerate by re-running the training cells
# (If your team wants ONE specific checkpoint version-controlled, use
#  `git lfs track` on that single file instead of un-ignoring this folder.)
models/
runs/

# Standard Python / notebook noise
__pycache__/
*.pyc
.ipynb_checkpoints/
"""

with open("../.gitignore", "w") as f:
    f.write(gitignore_contents.strip() + "\n")

print("Wrote .gitignore -- data/ and models/ will not be tracked by Git.")

Wrote .gitignore -- data/ and models/ will not be tracked by Git.


## PAIR 1 — Preprocessing

**Job:** turn a country-wide waterways file and a satellite composite into
two small, clean, buffer-clipped outputs.

**Downloads in this section:** OSM waterways shapefile, Sentinel-2 composite
from Earth Engine.

**Hands off to Pair 2 (HANDOFF):**
- `data/processed/kasarani_60m_riparian_zone.geojson`
- `data/processed/kasarani_composite_clipped.tif`
- `data/tiles/*.png`

In [6]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box, LineString

DIRS = ["../data/raw", "../data/processed", "../data/vectors", "../data/tiles", "../models", "../runs"]
for d in DIRS:
    os.makedirs(d, exist_ok=True)

# Reproducibility: any river corridor can be added here as {name, bbox}
# every downstream cell loops over REGIONS instead of a single hardcoded
# area.
# tile/sample counts scale with number of regions, not with one thin 60m
# buffer's area. Real bboxes below were sourced by searching the actual
# downloaded OSM waterways data for named river corridors .
REGIONS = [
    {"name": "kasarani",   "bbox": (36.80, -1.32, 36.95, -1.20)},
    {"name": "gatharaini", "bbox": (36.8984, -1.2522, 37.0221, -1.1930)},
    {"name": "motoine",    "bbox": (36.6702, -1.3346, 36.8105, -1.2803)},
]

print(f"Folders ready. {len(REGIONS)} regions configured: {[r['name'] for r in REGIONS]}")

Folders ready. 3 regions configured: ['kasarani', 'gatharaini', 'motoine']


---
## PAGE BREAK — Phase 1.A: Data Acquisition (download real OSM waterways)
---

### Step 1.1 — Download the waterways shapefile

The `gis_osm_waterways_free_1.shp` file used in this pipeline comes from
**Geofabrik** — a free provider of OpenStreetMap data, cut up by country.
It's a zip file containing several `.shp`-family files.

```
Source: https://download.geofabrik.de/africa/kenya-latest-free.shp.zip
```

We download it into `data/raw/` — which is already in `.gitignore`, so it
never touches your Git history.

**Trick:** you don't have to download this by hand every time. `requests` +
`zipfile` do it in a few lines, so any teammate can run this cell once after
cloning and get the exact same file. On a flaky connection, prefer a
resumable download (`curl -C -` or similar) over `requests.get()` in one
shot — a ~1GB file has more opportunities to hit a stalled connection.

In [7]:
import requests, zipfile, io

GEOFABRIK_URL = "https://download.geofabrik.de/africa/kenya-latest-free.shp.zip"
RAW_DIR = "../data/raw/kenya_osm"

def download_and_unzip(url, target_dir):
    if os.path.exists(target_dir) and os.listdir(target_dir):
        print(f"Already downloaded: {target_dir}")
        return
    os.makedirs(target_dir, exist_ok=True)
    print(f"Downloading {url} ...")
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
        zf.extractall(target_dir)
    print(f"Extracted into {target_dir}")

download_and_unzip(GEOFABRIK_URL, RAW_DIR)
VECTOR_INPUT = os.path.join(RAW_DIR, "gis_osm_waterways_free_1.shp")

Extracted into ../data/raw/kenya_osm


---
## PAGE BREAK — Phase 1.B: Vector Preprocessing (build the 60m legal riparian buffer)
---

### Step 1.2 — Build the legal riparian buffer

**Quick lesson: why reproject to EPSG:32737 before buffering?**

`EPSG:4326` (plain latitude/longitude) measures distance in *degrees*, and a
degree of longitude covers a different physical distance depending on how
far you are from the equator. Buffering "by 60" in that CRS gives you a
wobbly, geographically wrong distance. `EPSG:32737` (UTM Zone 37S) is a
*projected* CRS that measures in real meters for this part of the world —
Nairobi sits right inside it. So: reproject to meters, buffer by a real
60m, then reproject back to lat/lon for everyone else downstream to use.

In [8]:
def execute_vector_preprocessing(input_shapefile, region_bbox, output_vector_path, buffer_meters=60):
    if not os.path.exists(input_shapefile):
        print("WARNING: shapefile not found -- using a small mock river for now.")
        mock_line = LineString([(36.8219, -1.2921), (36.8350, -1.2850), (36.8500, -1.2900)])
        raw_gdf = gpd.GeoDataFrame(geometry=[mock_line], crs="EPSG:4326")
        raw_gdf["fclass"] = "river"
    else:
        raw_gdf = gpd.read_file(input_shapefile, bbox=box(*region_bbox))

    if "fclass" in raw_gdf.columns:
        filtered_rivers = raw_gdf[raw_gdf["fclass"].isin(["river", "stream"])].copy()
    else:
        filtered_rivers = raw_gdf.copy()

    rivers_metric = filtered_rivers.to_crs(epsg=32737)   # degrees -> real meters
    buffer_metric = rivers_metric.buffer(buffer_meters)
    buffer_gdf = gpd.GeoDataFrame(geometry=buffer_metric, crs="EPSG:32737")
    buffer_global = buffer_gdf.to_crs(epsg=4326)          # meters -> back to lat/lon

    buffer_global.to_file(output_vector_path, driver="GeoJSON")
    return buffer_global, len(filtered_rivers)

# Run once per region -- results keyed by region name so every later step
# can look up "this region's buffer/composite/tiles" by name.
riparian_boundaries = {}
for region in REGIONS:
    name = region["name"]
    out_path = f"../data/processed/{name}_60m_riparian_zone.geojson"
    print(f"[Pair 1] [{name}] Building the riparian buffer...")
    boundary, n_features = execute_vector_preprocessing(VECTOR_INPUT, region["bbox"], out_path)
    riparian_boundaries[name] = boundary
    region["vector_output"] = out_path
    print(f"[{name}] {n_features} river/stream features -> buffer saved: {out_path}")

[Pair 1] [kasarani] Building the riparian buffer...
[kasarani] 188 river/stream features -> buffer saved: ../data/processed/kasarani_60m_riparian_zone.geojson
[Pair 1] [gatharaini] Building the riparian buffer...
[gatharaini] 101 river/stream features -> buffer saved: ../data/processed/gatharaini_60m_riparian_zone.geojson
[Pair 1] [motoine] Building the riparian buffer...
[motoine] 85 river/stream features -> buffer saved: ../data/processed/motoine_60m_riparian_zone.geojson


---
## PAGE BREAK — Phase 1.C: Data Acquisition (pull the real Sentinel-2 composite)
---

### Step 1.3 — Download the Sentinel-2 composite (Earth Engine)

Two things happen here:
1. Build a cloud-filtered composite over the study area, using the **most
   recent available imagery** — not a fixed historical date range.
2. Export it straight to a local file with `geemap`, which skips the
   classic "export to Drive, then download separately" round-trip — much
   friendlier for a small AOI like this one.

**Why "most recent" instead of a fixed wet-season window:** an earlier
draft of this notebook hardcoded a wet-season date range (e.g.
"2026-03-01" to "2026-05-31"). That's the wrong default for a monitoring
tool — every re-run would show the same stale season instead of current
conditions. Below we instead take a rolling window ending *today*, sorted
by cloud cover, so the notebook always reflects what's actually there when
you run it. If your team specifically wants a seasonal comparison later,
make that an explicit, separate parameter rather than the default path.

**One-time setup, if you haven't already:** `ee.Authenticate()` opens a
browser login the first time; after that it's cached locally, so this cell
is safe to re-run.

**The project requirement:** newer versions of the Earth Engine API require
an explicit Google Cloud project tied to your account — `ee.Initialize()`
with no arguments now fails with "no project found." Find yours at
https://code.earthengine.google.com/ (shown in the project selector), or
register one at https://console.cloud.google.com/earth-engine if you don't
have one yet. Set it via the `EE_PROJECT_ID` environment variable, or edit
the fallback default in the cell below.

**Size constraint to know about:** Earth Engine's direct-download path
(`getDownloadURL`, what `geemap.ee_export_image` uses under the hood) caps
a single request at 48MB of raw pixel data. At native 10m resolution this
AOI's 3-band composite comes out to ~60MB and the export fails outright —
that's why `scale` below is 15m, not 10m. For a larger AOI or finer
resolution, you'd need `ee.batch.Export.image.toDrive` / `toCloudStorage`
instead (asynchronous tasks, no size cap) — worth switching to if you scale
this past one small suburb.

In [9]:
import os
import datetime
import ee
import geemap

ee.Authenticate()          # only prompts the first time

# Find your own project at https://code.earthengine.google.com/
EE_PROJECT_ID = os.environ.get("EE_PROJECT_ID", "solar-haven-349708")
ee.Initialize(project=EE_PROJECT_ID)

RECENT_DAYS = 90  # rolling window ending today, not a fixed historical range

def pull_recent_composite(region_bbox, out_path, scale=15):
    aoi = ee.Geometry.BBox(*region_bbox)
    end_date = ee.Date(datetime.date.today().isoformat())
    start_date = end_date.advance(-RECENT_DAYS, "day")

    recent_collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
        .sort("CLOUDY_PIXEL_PERCENTAGE")
    )
    n_scenes = recent_collection.size().getInfo()
    if n_scenes == 0:
        raise RuntimeError(
            f"No scenes under 20% cloud in the last {RECENT_DAYS} days for this "
            "region -- increase RECENT_DAYS or relax the cloud threshold."
        )

    composite = recent_collection.median().select(["B4", "B3", "B2"]).clip(aoi)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    geemap.ee_export_image(
        composite,
        filename=out_path,
        scale=scale,  # 10m native resolution exceeds the 48MB direct-download cap for these AOIs
        region=aoi,
    )
    return n_scenes

for region in REGIONS:
    name = region["name"]
    out_path = f"../data/raw/recent_composite/{name}_composite.tif"
    print(f"[{name}] pulling most-recent low-cloud Sentinel-2 composite...")
    n_scenes = pull_recent_composite(region["bbox"], out_path)
    region["composite_path"] = out_path
    print(f"[{name}] {n_scenes} scene(s) used -> {out_path}")

[kasarani] pulling most-recent low-cloud Sentinel-2 composite...
Generating URL ...
Please wait ...
Data downloaded to /home/miringu/Documents/data science/Moringa/Module6/lala_lala/data/raw/recent_composite/kasarani_composite.tif
[kasarani] 6 scene(s) used -> ../data/raw/recent_composite/kasarani_composite.tif
[gatharaini] pulling most-recent low-cloud Sentinel-2 composite...
Generating URL ...
Please wait ...
Data downloaded to /home/miringu/Documents/data science/Moringa/Module6/lala_lala/data/raw/recent_composite/gatharaini_composite.tif
[gatharaini] 6 scene(s) used -> ../data/raw/recent_composite/gatharaini_composite.tif
[motoine] pulling most-recent low-cloud Sentinel-2 composite...
Generating URL ...
Please wait ...
Data downloaded to /home/miringu/Documents/data science/Moringa/Module6/lala_lala/data/raw/recent_composite/motoine_composite.tif
[motoine] 14 scene(s) used -> ../data/raw/recent_composite/motoine_composite.tif


### Step 1.4 — Check the file size before it becomes a Git problem

A small habit that saves a rejected push later: check any raster's size the
moment it lands on disk, not the moment you try to commit it.

In [10]:
def check_github_size(path, warn_mb=50, block_mb=100):
    size_mb = os.path.getsize(path) / (1024 * 1024)
    if size_mb >= block_mb:
        print(f"BLOCKED: {path} is {size_mb:.1f}MB -- GitHub will refuse this push. Keep it .gitignore'd or use Git LFS.")
    elif size_mb >= warn_mb:
        print(f"WARNING: {path} is {size_mb:.1f}MB -- under the push limit, but worth Git LFS if you ever need to commit it.")
    else:
        print(f"OK: {path} is {size_mb:.1f}MB -- safe to commit if you ever needed to.")
    return size_mb

for region in REGIONS:
    path = region.get("composite_path")
    if path and os.path.exists(path):
        check_github_size(path)

OK: ../data/raw/recent_composite/kasarani_composite.tif is 6.1MB -- safe to commit if you ever needed to.
OK: ../data/raw/recent_composite/gatharaini_composite.tif is 2.5MB -- safe to commit if you ever needed to.
OK: ../data/raw/recent_composite/motoine_composite.tif is 2.6MB -- safe to commit if you ever needed to.


---
## PAGE BREAK — Phase 1.D: Raster Preprocessing (size check, clip to buffer, tile, 8-bit stretch)
---

### Step 1.5 — Clip to the buffer and cut into tiles

Two reasons this step exists: (1) Pair 2 shouldn't load pixels from outside
the legal buffer zone — wastes RAM and pollutes training data; (2) YOLOv8
expects individual image files, not one giant raster, so we cut the clipped
image into fixed-size tiles.

In [11]:
def clip_and_tile_satellite_composite(source_raster_path, riparian_buffer_geojson_path,
                                       clipped_output_path, tile_output_dir, tile_size=640):
    import rasterio
    from rasterio.mask import mask
    from PIL import Image

    buffer_gdf = gpd.read_file(riparian_buffer_geojson_path)

    with rasterio.open(source_raster_path) as src:
        if buffer_gdf.crs != src.crs:
            buffer_gdf = buffer_gdf.to_crs(src.crs)
        geoms = [g.__geo_interface__ for g in buffer_gdf.geometry]
        clipped_array, clipped_transform = mask(src, geoms, crop=True)
        clipped_meta = src.meta.copy()
        clipped_meta.update({
            "height": clipped_array.shape[1],
            "width": clipped_array.shape[2],
            "transform": clipped_transform,
        })

    with rasterio.open(clipped_output_path, "w", **clipped_meta) as dst:
        dst.write(clipped_array)

    os.makedirs(tile_output_dir, exist_ok=True)
    bands, height, width = clipped_array.shape
    rgb = clipped_array[:3] if bands >= 3 else clipped_array
    tile_count = 0
    for row in range(0, height, tile_size):
        for col in range(0, width, tile_size):
            tile = rgb[:, row:row + tile_size, col:col + tile_size]
            if tile.shape[1] < 10 or tile.shape[2] < 10:
                continue
            tile_img = np.moveaxis(tile, 0, -1)
            tile_img = np.clip(tile_img, 0, 255).astype("uint8")
            Image.fromarray(tile_img).save(os.path.join(tile_output_dir, f"tile_{tile_count:04d}.png"))
            tile_count += 1
    return clipped_output_path, tile_count

for region in REGIONS:
    name = region["name"]
    composite_path = region.get("composite_path")
    if not composite_path or not os.path.exists(composite_path):
        print(f"[{name}] skipping clip/tile -- no composite found")
        continue
    clipped_path = f"../data/processed/{name}_composite_clipped.tif"
    tile_dir = f"../data/tiles/{name}"
    _, tile_count = clip_and_tile_satellite_composite(composite_path, region["vector_output"], clipped_path, tile_dir)
    region["clipped_path"] = clipped_path
    region["tile_dir"] = tile_dir
    region["tile_count"] = tile_count
    print(f"[{name}] clipped -> {clipped_path}; {tile_count} tiles -> {tile_dir}/")

[kasarani] clipped -> ../data/processed/kasarani_composite_clipped.tif; 4 tiles -> ../data/tiles/kasarani/
[gatharaini] clipped -> ../data/processed/gatharaini_composite_clipped.tif; 2 tiles -> ../data/tiles/gatharaini/
[motoine] clipped -> ../data/processed/motoine_composite_clipped.tif; 2 tiles -> ../data/tiles/motoine/


### Step 1.6 — Rescale to 8-bit before any pixel-value logic

**A second real-data bug, found by actually running this on live Sentinel-2
data instead of the mock fallback:** Sentinel-2 surface-reflectance bands
are NOT 0-255 like a normal photo -- they're unscaled integers that
commonly run 0-13,000+ (median around 1,000-1,500 for this AOI). Both of
the following break silently if you feed them raw reflectance:
- Step 2.1's roof/water/vegetation thresholds (`r_val > 150`, etc.) assume
  0-255, so almost every pixel would clear 150 and get misclassified as
  "roof."
- The GLCM texture function casts the grayscale patch to `uint8`, which
  wraps/truncates values above 255 into meaningless texture stats.

Fix: apply the standard Sentinel-2 true-color stretch (clip reflectance to
[0, 3000], scale to [0, 255]) once here, and have every pixel-level step
downstream read from the stretched raster instead of the raw clipped one.

In [12]:
import numpy as np
import rasterio

def rescale_to_8bit(input_path, output_path, reflectance_max=3000):
    with rasterio.open(input_path) as src:
        arr = src.read().astype(np.float32)
        meta = src.meta.copy()

    stretched = np.clip(arr / reflectance_max * 255.0, 0, 255).astype(np.uint8)
    meta.update(dtype="uint8")

    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(stretched)
    return output_path

for region in REGIONS:
    name = region["name"]
    clipped_path = region.get("clipped_path")
    if not clipped_path or not os.path.exists(clipped_path):
        print(f"[{name}] skipping rescale -- no clipped raster")
        continue
    stretched_path = f"../data/processed/{name}_composite_clipped_8bit.tif"
    rescale_to_8bit(clipped_path, stretched_path)
    region["stretched_path"] = stretched_path
    print(f"[{name}] 8-bit stretched raster saved: {stretched_path}")

[kasarani] 8-bit stretched raster saved: ../data/processed/kasarani_composite_clipped_8bit.tif
[gatharaini] 8-bit stretched raster saved: ../data/processed/gatharaini_composite_clipped_8bit.tif
[motoine] 8-bit stretched raster saved: ../data/processed/motoine_composite_clipped_8bit.tif
